# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_14 — Local Volatility and the Dupire Surface

### Research question

How can an implied volatility surface be converted into a local volatility surface using Dupire's formula, and how stable is that transformation numerically?

This notebook extends:

```text
02_09_volatility_surface_construction
02_10_finite_difference_pde_pricer
02_11_heston_model_calibration
02_12_sabr_smile_calibration
```

The previous notebooks constructed and calibrated implied volatility surfaces. This notebook turns a surface into a **state-dependent volatility model**:

$$
dS_t = (r-q)S_tdt + \sigma_{\text{loc}}(t,S_t)S_tdW_t
$$
It covers:

1. implied volatility surface generation;
2. conversion from implied volatilities to call prices;
3. static-arbitrage diagnostics;
4. Dupire local volatility formula;
5. numerical derivatives with respect to strike and maturity;
6. local variance stability checks;
7. smoothing and clipping diagnostics;
8. local volatility heatmaps;
9. local-volatility Monte Carlo repricing;
10. model-vs-surface validation;
11. limitations and research-frontier extensions.

The key idea:

> Dupire local volatility converts a full option surface into a diffusion model that is designed to match European vanilla option prices across strikes and maturities.

## 1. From implied volatility to local volatility

A Black-Scholes implied volatility surface is a quoting convention:

$$
(K,T)\mapsto \sigma_{\text{imp}}(K,T)
$$
A local volatility model is a stochastic process:

$$
dS_t=(r-q)S_tdt+\sigma_{\text{loc}}(t,S_t)S_tdW_t
$$
The local volatility function is chosen so that the model reproduces the observed European option surface.

This is different from Heston or SABR:

| Model | Main object | Use |
|---|---|---|
| BSM IV surface | $\sigma_{\text{imp}}(K,T)$ | Market quote representation |
| Heston | stochastic variance process | Dynamic stochastic-vol model |
| SABR | forward smile parameterisation | Market smile fitting |
| Dupire local vol | $\sigma_{\text{loc}}(t,S)$ | Surface-consistent diffusion |

Dupire is powerful because it converts cross-sectional option prices into a diffusion coefficient.

## 2. Dupire formula

Let $C(K,T)$ be the European call price as a function of strike and maturity.

With continuous risk-free rate $r$ and dividend yield $q$, the local variance is:

$$
\begin{aligned}
\sigma_{\text{loc}}^2(K,T)
&= \frac{C_T + qC + (r-q)K C_K}{\frac{1}{2}K^2 C_{KK}} \\
&= \frac{\frac{\partial C}{\partial T} + qC + (r-q)K\frac{\partial C}{\partial K}}{\frac{1}{2}K^2\frac{\partial^2 C}{\partial K^2}}
\end{aligned}
$$
The denominator is related to the risk-neutral density:

$$
\begin{aligned}
\frac{\partial^2 C}{\partial K^2} &= e^{-rT} f_{S_T}(K)
\end{aligned}
$$
Therefore, if call prices are not convex in strike, the local volatility calculation can become unstable or invalid.

This is why local volatility construction requires a smooth, arbitrage-aware surface.

## 3. Why this is numerically delicate

Dupire requires derivatives of option prices:

$$
C_T,\quad C_K,\quad C_{KK}
$$
Numerical differentiation amplifies noise.

Small quote errors in implied volatility can become large errors in local volatility.

Common problems:

- negative call convexity;
- calendar arbitrage;
- noisy wings;
- sparse strikes;
- bad interpolation;
- small denominator near illiquid regions;
- explosive local volatility in extrapolation zones.

The engineering lesson:

> Local volatility is only as good as the smoothness and arbitrage quality of the input surface.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class DupireConfig:
    spot: float = 100.0
    risk_free_rate: float = 0.035
    dividend_yield: float = 0.005
    min_local_vol: float = 0.02
    max_local_vol: float = 1.50
    seed: int = 42


config = DupireConfig()
config

## 5. Black-Scholes helpers

We use Black-Scholes to convert the synthetic implied volatility surface into call prices.

The local volatility calculation uses call prices, not implied volatilities directly.

In [ ]:
def normal_cdf(x):
    x = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def normal_pdf(x):
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5 * x**2) / np.sqrt(2.0 * np.pi)


def bsm_price(spot, strike, maturity, r, q, vol, option_type="call"):
    spot = np.asarray(spot, dtype=float)
    strike = np.asarray(strike, dtype=float)

    if maturity <= 0:
        if option_type == "call":
            return np.maximum(spot - strike, 0.0)
        if option_type == "put":
            return np.maximum(strike - spot, 0.0)
        raise ValueError("option_type must be call or put")

    spot_safe = np.maximum(spot, 1e-12)
    strike_safe = np.maximum(strike, 1e-12)

    d1 = (
        np.log(spot_safe / strike_safe)
        + (r - q + 0.5 * vol**2) * maturity
    ) / (vol * np.sqrt(maturity))

    d2 = d1 - vol * np.sqrt(maturity)

    discounted_spot = spot * np.exp(-q * maturity)
    discounted_strike = strike * np.exp(-r * maturity)

    if option_type == "call":
        return discounted_spot * normal_cdf(d1) - discounted_strike * normal_cdf(d2)
    if option_type == "put":
        return discounted_strike * normal_cdf(-d2) - discounted_spot * normal_cdf(-d1)

    raise ValueError("option_type must be call or put")


def implied_vol_bisection(price, spot, strike, maturity, r, q, option_type="call", lo=1e-6, hi=4.0):
    """
    Simple implied vol inversion for validation.
    """
    if maturity <= 0:
        return np.nan

    discounted_spot = spot * exp(-q * maturity)
    discounted_strike = strike * exp(-r * maturity)

    if option_type == "call":
        lower = max(discounted_spot - discounted_strike, 0.0)
        upper = discounted_spot
    else:
        lower = max(discounted_strike - discounted_spot, 0.0)
        upper = discounted_strike

    if price < lower - 1e-8 or price > upper + 1e-8:
        return np.nan

    def objective(vol):
        return float(bsm_price(spot, strike, maturity, r, q, vol, option_type)) - price

    f_lo = objective(lo)
    f_hi = objective(hi)

    if f_lo * f_hi > 0:
        return np.nan

    for _ in range(120):
        mid = 0.5 * (lo + hi)
        f_mid = objective(mid)
        if abs(f_mid) < 1e-10 or hi - lo < 1e-10:
            return float(mid)

        if f_lo * f_mid <= 0:
            hi = mid
            f_hi = f_mid
        else:
            lo = mid
            f_lo = f_mid

    return float(0.5 * (lo + hi))


def forward_price(spot, maturity, r, q):
    return spot * exp((r - q) * maturity)

## 6. Synthetic implied volatility surface

We create a smooth synthetic implied volatility surface in log-forward-moneyness:

$$
k=\log(K/F_T)
$$
The surface includes:

- ATM term structure;
- downside skew;
- smile curvature;
- stronger short-maturity curvature.

This is designed to be smooth enough for Dupire differentiation.

In [ ]:
def synthetic_implied_vol(spot, strike, maturity, r, q):
    F = forward_price(spot, maturity, r, q)
    k = log(strike / F)

    atm = 0.18 + 0.05 * exp(-2.0 * maturity) + 0.02 * (1 - exp(-0.8 * maturity))
    skew = -0.12 * exp(-0.5 * maturity) * k
    curvature = (0.35 + 0.18 * exp(-2.0 * maturity)) * k**2
    wing = 0.04 * abs(k)**1.5

    vol = atm + skew + curvature + wing
    return float(np.clip(vol, 0.05, 1.50))


K_grid = np.linspace(55.0, 155.0, 101)
T_grid = np.array([14, 21, 30, 45, 60, 90, 120, 180, 270, 365, 540, 730]) / 365.0

surface_rows = []

for T in T_grid:
    F = forward_price(config.spot, T, config.risk_free_rate, config.dividend_yield)
    for K in K_grid:
        iv = synthetic_implied_vol(config.spot, K, T, config.risk_free_rate, config.dividend_yield)
        call = float(bsm_price(config.spot, K, T, config.risk_free_rate, config.dividend_yield, iv, "call"))
        surface_rows.append({
            "maturity": float(T),
            "strike": float(K),
            "forward": F,
            "log_moneyness": log(K / F),
            "implied_vol": iv,
            "total_variance": iv**2 * T,
            "call_price": call
        })

surface = pd.DataFrame(surface_rows)

surface.head()

In [ ]:
iv_pivot = surface.pivot(index="maturity", columns="strike", values="implied_vol")

plt.figure(figsize=(12, 6))
plt.imshow(
    iv_pivot.values,
    origin="lower",
    aspect="auto",
    extent=[K_grid.min(), K_grid.max(), T_grid.min(), T_grid.max()]
)
plt.colorbar(label="Implied volatility")
plt.title("Synthetic Implied Volatility Surface")
plt.xlabel("Strike")
plt.ylabel("Maturity")
plt.show()

## 7. Call price surface diagnostics

Dupire requires a call price surface.

Before differentiating, we check:

1. call prices decrease in strike;
2. call prices are convex in strike;
3. call prices generally increase with maturity for comparable strikes.

Convexity is especially important because:

$$
C_{KK}
$$
appears in the denominator of Dupire's formula.

In [ ]:
def static_arbitrage_diagnostics(call_surface: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for T, group in call_surface.groupby("maturity"):
        g = group.sort_values("strike")
        C = g["call_price"].to_numpy()
        K = g["strike"].to_numpy()

        first_diff = np.diff(C)
        second_diff = C[:-2] - 2 * C[1:-1] + C[2:]

        rows.append({
            "maturity": float(T),
            "n_strikes": len(g),
            "monotonicity_violations": int(np.sum(first_diff > 1e-8)),
            "convexity_violations": int(np.sum(second_diff < -1e-8)),
            "min_first_diff": float(first_diff.min()),
            "min_second_diff": float(second_diff.min())
        })

    return pd.DataFrame(rows)


arb_report = static_arbitrage_diagnostics(surface)

arb_report

## 8. Numerical derivatives

We reshape the call price grid into a matrix:

$$
C_{j,i}=C(T_j,K_i)
$$
Then compute:

$$
C_T
$$
by finite differences across maturity and:

$$
C_K,\quad C_{KK}
$$
by finite differences across strike.

We use `numpy.gradient`, which handles nonuniform grids if supplied the grid values.

In [ ]:
C_grid = surface.pivot(index="maturity", columns="strike", values="call_price").sort_index().sort_index(axis=1)
IV_grid = surface.pivot(index="maturity", columns="strike", values="implied_vol").sort_index().sort_index(axis=1)

T_values = C_grid.index.to_numpy()
K_values = C_grid.columns.to_numpy()
C_values = C_grid.to_numpy()

# Axis 0 = maturity, axis 1 = strike.
C_T = np.gradient(C_values, T_values, axis=0, edge_order=2)
C_K = np.gradient(C_values, K_values, axis=1, edge_order=2)
C_KK = np.gradient(C_K, K_values, axis=1, edge_order=2)

derivative_rows = []

for ti, T in enumerate(T_values):
    for ki, K in enumerate(K_values):
        derivative_rows.append({
            "maturity": float(T),
            "strike": float(K),
            "call_price": float(C_values[ti, ki]),
            "C_T": float(C_T[ti, ki]),
            "C_K": float(C_K[ti, ki]),
            "C_KK": float(C_KK[ti, ki]),
            "implied_vol": float(IV_grid.iloc[ti, ki])
        })

derivatives = pd.DataFrame(derivative_rows)

derivatives.head()

## 9. Dupire local volatility calculation

Dupire local variance is:

$$
\begin{aligned}
\sigma_{\text{loc}}^2(K,T) &= \frac{ C_T+qC+(r-q)KC_K } { \frac12 K^2C_{KK} }
\end{aligned}
$$
We compute the numerator, denominator, raw local variance, and clipped local volatility.

Invalid regions are flagged rather than silently trusted.

In [ ]:
def compute_dupire_local_vol(derivatives: pd.DataFrame, config: DupireConfig) -> pd.DataFrame:
    out = derivatives.copy()

    r = config.risk_free_rate
    q = config.dividend_yield

    out["dupire_numerator"] = (
        out["C_T"]
        + q * out["call_price"]
        + (r - q) * out["strike"] * out["C_K"]
    )

    out["dupire_denominator"] = 0.5 * out["strike"]**2 * out["C_KK"]

    out["raw_local_variance"] = out["dupire_numerator"] / out["dupire_denominator"]

    out["valid_density"] = out["C_KK"] > 1e-10
    out["valid_numerator"] = out["dupire_numerator"] > 0
    out["valid_local_variance"] = (
        np.isfinite(out["raw_local_variance"])
        & (out["raw_local_variance"] > 0)
        & out["valid_density"]
        & out["valid_numerator"]
    )

    out["local_variance"] = out["raw_local_variance"].where(out["valid_local_variance"])
    out["local_vol"] = np.sqrt(out["local_variance"])

    out["local_vol_clipped"] = out["local_vol"].clip(
        lower=config.min_local_vol,
        upper=config.max_local_vol
    )

    out["local_vol_was_clipped"] = (
        out["local_vol"].notna()
        & (
            (out["local_vol"] < config.min_local_vol)
            | (out["local_vol"] > config.max_local_vol)
        )
    )

    return out


local_vol_surface = compute_dupire_local_vol(derivatives, config)

local_vol_surface.head()

In [ ]:
local_vol_quality = pd.Series({
    "n_points": len(local_vol_surface),
    "valid_local_variance_fraction": local_vol_surface["valid_local_variance"].mean(),
    "invalid_density_count": int((~local_vol_surface["valid_density"]).sum()),
    "invalid_numerator_count": int((~local_vol_surface["valid_numerator"]).sum()),
    "local_vol_clip_count": int(local_vol_surface["local_vol_was_clipped"].sum()),
    "min_raw_local_vol": float(local_vol_surface["local_vol"].min(skipna=True)),
    "max_raw_local_vol": float(local_vol_surface["local_vol"].max(skipna=True)),
    "median_raw_local_vol": float(local_vol_surface["local_vol"].median(skipna=True))
})

local_vol_quality

## 10. Local volatility heatmap

A local volatility surface is indexed by maturity and spot/strike level.

Under the local-volatility interpretation, the strike axis becomes a state axis:

$$
K \leftrightarrow S
$$
So $\sigma_{\text{loc}}(T,K)$ can be read as:

$$
\sigma_{\text{loc}}(t,S)
$$

In [ ]:
lv_pivot = local_vol_surface.pivot(index="maturity", columns="strike", values="local_vol_clipped")

plt.figure(figsize=(12, 6))
plt.imshow(
    lv_pivot.values,
    origin="lower",
    aspect="auto",
    extent=[K_values.min(), K_values.max(), T_values.min(), T_values.max()]
)
plt.colorbar(label="Local volatility")
plt.title("Dupire Local Volatility Surface")
plt.xlabel("Strike / state level")
plt.ylabel("Maturity / time")
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))

for T, group in local_vol_surface.groupby("maturity"):
    if T in T_values[::2]:
        group = group.sort_values("strike")
        plt.plot(group["strike"], group["local_vol_clipped"], label=f"T={T:.2f}")

plt.title("Local Volatility Slices")
plt.xlabel("Strike / state level")
plt.ylabel("Local volatility")
plt.legend()
plt.show()

## 11. Compare implied volatility and local volatility

Implied volatility and local volatility are not the same object.

- implied volatility is a Black-Scholes quote for an option with strike $K$ and maturity $T$;
- local volatility is the instantaneous volatility used when the process is at state $S$ and time $t$.

The two surfaces can look quite different.

In [ ]:
comparison_rows = []

for row in local_vol_surface.itertuples():
    comparison_rows.append({
        "maturity": row.maturity,
        "strike": row.strike,
        "implied_vol": row.implied_vol,
        "local_vol": row.local_vol_clipped,
        "local_minus_implied": row.local_vol_clipped - row.implied_vol
    })

iv_lv_comparison = pd.DataFrame(comparison_rows)

iv_lv_comparison["local_minus_implied"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(iv_lv_comparison["implied_vol"], iv_lv_comparison["local_vol"], alpha=0.4, s=10)
min_v = min(iv_lv_comparison["implied_vol"].min(), iv_lv_comparison["local_vol"].min())
max_v = max(iv_lv_comparison["implied_vol"].max(), iv_lv_comparison["local_vol"].max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
plt.title("Implied Volatility vs Dupire Local Volatility")
plt.xlabel("Implied volatility")
plt.ylabel("Local volatility")
plt.show()

## 12. Interpolating the local volatility surface

To simulate a local volatility process, we need:

$$
\sigma_{\text{loc}}(t,S_t)
$$
for arbitrary $t$ and $S_t$.

We implement bilinear interpolation on the local volatility grid.

In production, extrapolation must be handled very carefully. Here, we clip state and time to the grid bounds.

In [ ]:
def prepare_local_vol_grid(local_vol_df: pd.DataFrame) -> dict:
    """
    Prepare a sorted local-volatility grid once, so simulation does not rebuild
    a pandas pivot table inside every path/time-step interpolation.
    """
    pivot = (
        local_vol_df
        .pivot(index="maturity", columns="strike", values="local_vol_clipped")
        .sort_index()
        .sort_index(axis=1)
    )

    return {
        "maturities": pivot.index.to_numpy(dtype=float),
        "strikes": pivot.columns.to_numpy(dtype=float),
        "values": pivot.to_numpy(dtype=float),
    }


def bilinear_local_vol(local_vol_grid: dict, T, S):
    """
    Bilinear interpolation of local volatility on a maturity/strike grid.

    `S` can be either a scalar or a NumPy array. Vectorising this function is
    important because the Monte Carlo validation calls it for many paths.
    """
    T_grid_interp = local_vol_grid["maturities"]
    K_grid_interp = local_vol_grid["strikes"]
    V = local_vol_grid["values"]

    T_clipped = float(np.clip(T, T_grid_interp.min(), T_grid_interp.max()))
    S_array = np.asarray(S, dtype=float)
    scalar_input = S_array.ndim == 0
    S_flat = np.atleast_1d(S_array).astype(float)
    S_clipped = np.clip(S_flat, K_grid_interp.min(), K_grid_interp.max())

    T_idx = np.searchsorted(T_grid_interp, T_clipped, side="right") - 1
    T_idx = int(np.clip(T_idx, 0, len(T_grid_interp) - 2))

    K_idx = np.searchsorted(K_grid_interp, S_clipped, side="right") - 1
    K_idx = np.clip(K_idx, 0, len(K_grid_interp) - 2)

    T0 = T_grid_interp[T_idx]
    T1 = T_grid_interp[T_idx + 1]
    K0 = K_grid_interp[K_idx]
    K1 = K_grid_interp[K_idx + 1]

    wt = 0.0 if T1 == T0 else (T_clipped - T0) / (T1 - T0)
    wk = np.divide(
        S_clipped - K0,
        K1 - K0,
        out=np.zeros_like(S_clipped, dtype=float),
        where=(K1 != K0),
    )

    v00 = V[T_idx, K_idx]
    v01 = V[T_idx, K_idx + 1]
    v10 = V[T_idx + 1, K_idx]
    v11 = V[T_idx + 1, K_idx + 1]

    interpolated = (
        (1 - wt) * (1 - wk) * v00
        + (1 - wt) * wk * v01
        + wt * (1 - wk) * v10
        + wt * wk * v11
    )

    if scalar_input:
        return float(interpolated[0])
    return interpolated.reshape(S_array.shape)


local_vol_grid = prepare_local_vol_grid(local_vol_surface)

# Sanity checks.
pd.Series({
    "lv_at_3m_90": bilinear_local_vol(local_vol_grid, 0.25, 90.0),
    "lv_at_1y_100": bilinear_local_vol(local_vol_grid, 1.0, 100.0),
    "lv_at_2y_120": bilinear_local_vol(local_vol_grid, 2.0, 120.0)
})


## 13. Local-volatility Monte Carlo

We simulate the local volatility SDE:

$$
dS_t=(r-q)S_tdt+\sigma_{\text{loc}}(t,S_t)S_tdW_t
$$
using a log-Euler style update:

$$
\begin{aligned}
S_{t+\Delta t} &= S_t \exp \Bigg[ (r-q-\frac12\sigma_{\text{loc}}^2)\Delta t \\
&\quad + \sigma_{\text{loc}}\sqrt{\Delta t}Z \Bigg]
\end{aligned}
$$
This is an approximation. The purpose here is validation: do prices from the local-vol model roughly reproduce the input vanilla surface?

In [ ]:
def simulate_local_vol_paths(config: DupireConfig, local_vol_grid: dict, maturity, n_steps, n_paths, seed=123):
    rng = np.random.default_rng(seed)
    dt = maturity / n_steps

    paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = config.spot

    min_grid_maturity = float(local_vol_grid["maturities"].min())

    for step in range(n_steps):
        t = max(step * dt, min_grid_maturity)
        S_now = paths[:, step]

        vols = bilinear_local_vol(local_vol_grid, t, S_now)
        z = rng.standard_normal(n_paths)

        log_increment = (
            (config.risk_free_rate - config.dividend_yield - 0.5 * vols**2) * dt
            + vols * sqrt(dt) * z
        )

        paths[:, step + 1] = S_now * np.exp(log_increment)

    return paths


def mc_price_call_from_paths(paths, strike, maturity, r):
    payoff = np.maximum(paths[:, -1] - strike, 0.0)
    discounted = np.exp(-r * maturity) * payoff
    price = float(np.mean(discounted))
    se = float(np.std(discounted, ddof=1) / np.sqrt(len(discounted)))
    return price, se


## 14. Repricing validation

A perfect Dupire local-volatility model, with exact derivatives and exact numerical methods, should reproduce the input European option prices.

Our implementation is deliberately transparent and approximate, so we expect small errors from:

- finite-difference derivative noise;
- clipping;
- interpolation;
- Monte Carlo error;
- Euler discretisation error.

We validate a small grid of strikes and maturities.

In [ ]:
validation_maturities = np.array([90, 180, 365]) / 365.0
validation_strikes = np.array([80, 90, 100, 110, 120])

repricing_rows = []

for T in validation_maturities:
    paths = simulate_local_vol_paths(
        config=config,
        local_vol_grid=local_vol_grid,
        maturity=float(T),
        n_steps=max(40, int(252 * T)),
        n_paths=40_000,
        seed=int(10000 * T)
    )

    for K in validation_strikes:
        mc_price, mc_se = mc_price_call_from_paths(paths, K, float(T), config.risk_free_rate)

        input_iv = synthetic_implied_vol(config.spot, K, float(T), config.risk_free_rate, config.dividend_yield)
        input_price = float(bsm_price(
            config.spot,
            K,
            float(T),
            config.risk_free_rate,
            config.dividend_yield,
            input_iv,
            "call"
        ))

        model_iv = implied_vol_bisection(
            mc_price,
            config.spot,
            K,
            float(T),
            config.risk_free_rate,
            config.dividend_yield,
            "call"
        )

        repricing_rows.append({
            "maturity": float(T),
            "strike": float(K),
            "input_surface_iv": input_iv,
            "input_surface_price": input_price,
            "local_vol_mc_price": mc_price,
            "local_vol_mc_se": mc_se,
            "price_error": mc_price - input_price,
            "abs_price_error": abs(mc_price - input_price),
            "local_vol_mc_implied_vol": model_iv,
            "iv_error": model_iv - input_iv,
            "abs_iv_error": abs(model_iv - input_iv)
        })

repricing_report = pd.DataFrame(repricing_rows)

repricing_report

In [ ]:
repricing_summary = pd.Series({
    "mean_abs_price_error": repricing_report["abs_price_error"].mean(),
    "max_abs_price_error": repricing_report["abs_price_error"].max(),
    "mean_abs_iv_error": repricing_report["abs_iv_error"].mean(),
    "max_abs_iv_error": repricing_report["abs_iv_error"].max(),
    "mean_mc_standard_error": repricing_report["local_vol_mc_se"].mean()
})

repricing_summary

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(repricing_report["input_surface_price"], repricing_report["local_vol_mc_price"], alpha=0.8)
min_p = min(repricing_report["input_surface_price"].min(), repricing_report["local_vol_mc_price"].min())
max_p = max(repricing_report["input_surface_price"].max(), repricing_report["local_vol_mc_price"].max())
plt.plot([min_p, max_p], [min_p, max_p], linestyle="--")
plt.title("Input Surface Prices vs Local-Vol MC Prices")
plt.xlabel("Input surface call price")
plt.ylabel("Local-vol MC call price")
plt.show()

## 15. Local volatility diagnostics by region

Local volatility is often least stable in the wings and at very short maturities.

We create region labels:

- short maturity;
- middle maturity;
- long maturity;
- low strike wing;
- near ATM;
- high strike wing.

This helps identify where local vol is reliable and where it is extrapolation-sensitive.

In [ ]:
diagnostic = local_vol_surface.copy()

diagnostic["maturity_bucket"] = pd.cut(
    diagnostic["maturity"],
    bins=[0, 0.20, 0.75, 3.0],
    labels=["short", "medium", "long"]
)

diagnostic["strike_bucket"] = pd.cut(
    diagnostic["strike"],
    bins=[0, 80, 120, 10_000],
    labels=["left_wing", "near_atm", "right_wing"]
)

region_diagnostics = (
    diagnostic
    .groupby(["maturity_bucket", "strike_bucket"], observed=True)
    .agg(
        n_points=("local_vol_clipped", "count"),
        valid_fraction=("valid_local_variance", "mean"),
        mean_local_vol=("local_vol_clipped", "mean"),
        median_local_vol=("local_vol_clipped", "median"),
        max_local_vol=("local_vol_clipped", "max"),
        clip_fraction=("local_vol_was_clipped", "mean")
    )
    .reset_index()
)

region_diagnostics

## 16. Practical checklist for Dupire construction

Before trusting a local volatility surface, check:

1. **Input surface quality**  
   Is the implied volatility surface smooth and arbitrage-aware?

2. **Call price monotonicity**  
   Are call prices decreasing in strike?

3. **Call price convexity**  
   Is $C_{KK}\geq0$?

4. **Calendar consistency**  
   Are option prices or total variances consistent across maturity?

5. **Derivative stability**  
   Are $C_T$, $C_K$, and $C_{KK}$ numerically stable?

6. **Density denominator**  
   Is the denominator in Dupire positive and far from zero?

7. **Local variance positivity**  
   Is $\sigma_{\text{loc}}^2>0$?

8. **Wings and extrapolation**  
   Are wings clipped, extrapolated, or modelled explicitly?

9. **Repricing validation**  
   Does the local-vol model reproduce the input vanilla surface?

10. **Documentation**  
   Are smoothing, clipping, and invalid regions saved?

## 17. Saving outputs

The notebook saves:

1. synthetic implied volatility surface;
2. call price surface;
3. static arbitrage diagnostics;
4. numerical derivative grid;
5. Dupire local volatility surface;
6. quality diagnostics;
7. repricing validation report;
8. regional diagnostics;
9. manifest.

In [ ]:
output_dir = Path("data/processed/local_volatility_dupire_surface_v1")
output_dir.mkdir(parents=True, exist_ok=True)

surface_path = output_dir / "synthetic_implied_vol_and_call_surface.csv"
arb_report_path = output_dir / "static_arbitrage_diagnostics.csv"
derivatives_path = output_dir / "call_surface_derivatives.csv"
local_vol_path = output_dir / "dupire_local_vol_surface.csv"
quality_path = output_dir / "local_vol_quality.csv"
comparison_path = output_dir / "implied_vs_local_vol_comparison.csv"
repricing_path = output_dir / "local_vol_repricing_validation.csv"
region_diag_path = output_dir / "local_vol_region_diagnostics.csv"
manifest_path = output_dir / "manifest.json"

surface.to_csv(surface_path, index=False)
arb_report.to_csv(arb_report_path, index=False)
derivatives.to_csv(derivatives_path, index=False)
local_vol_surface.to_csv(local_vol_path, index=False)
local_vol_quality.to_frame("value").to_csv(quality_path)
iv_lv_comparison.to_csv(comparison_path, index=False)
repricing_report.to_csv(repricing_path, index=False)
region_diagnostics.to_csv(region_diag_path, index=False)

manifest = {
    "dataset_name": "local_volatility_dupire_surface_outputs",
    "schema_version": "local_volatility_dupire_surface_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "n_surface_points": int(len(surface)),
    "n_local_vol_points": int(len(local_vol_surface)),
    "local_vol_quality": local_vol_quality.to_dict(),
    "repricing_summary": repricing_summary.to_dict(),
    "notes": [
        "Synthetic implied volatility surface is converted to Black-Scholes call prices.",
        "Dupire local variance uses finite differences in maturity and strike.",
        "Invalid density or negative local variance regions are flagged.",
        "Local volatility is clipped only for simulation/interpolation diagnostics.",
        "Monte Carlo repricing is approximate and includes MC/discretisation error."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

surface_path, local_vol_path, repricing_path, manifest_path

## 18. Limitations

### 18.1 Synthetic surface

The input surface is synthetic and smooth.

Real market surfaces require quote cleaning, arbitrage-aware smoothing, and careful extrapolation.

### 18.2 Numerical derivatives amplify noise

Dupire uses $C_T$, $C_K$, and $C_{KK}$.  
These derivatives are unstable if the input surface is noisy.

### 18.3 Simple finite differences

This notebook uses transparent finite differences.

Production systems may use splines, SVI/SSVI, smoothing penalties, or constrained optimisation.

### 18.4 Local volatility is not stochastic volatility

Local volatility matches vanilla prices under ideal conditions, but it often produces unrealistic forward smile dynamics.

### 18.5 Extrapolation risk

The local-volatility function outside the liquid strike/maturity region can be unstable.

This matters for Monte Carlo paths that move into the wings.

### 18.6 Approximate repricing validation

The local-vol repricing uses Monte Carlo with interpolation and discretisation.

A PDE solver under the local-vol model would be a cleaner validation method.

### 18.7 No jumps

Dupire local volatility assumes continuous diffusion dynamics.

It cannot represent discontinuous jump risk directly.

## 19. Research frontier and extensions

Important extensions include:

1. **Arbitrage-free surface smoothing**  
   Build a smooth call surface with monotonicity and convexity constraints.

2. **SVI/SSVI to Dupire**  
   Use an arbitrage-aware implied volatility parameterisation before applying Dupire.

3. **Local volatility PDE pricer**  
   Modify the finite difference PDE pricer so $\sigma=\sigma_{\text{loc}}(t,S)$.

4. **Local-stochastic volatility**  
   Combine local volatility with stochastic volatility to match vanillas and improve smile dynamics.

5. **Dupire with rates/dividends curves**  
   Use full discount and forward curves rather than constant $r,q$.

6. **Forward-start smile diagnostics**  
   Test whether local vol produces realistic future smiles.

7. **GPU/local-vol Monte Carlo**  
   Accelerate interpolation-heavy local-vol path simulation.

8. **Chinese futures options extension**  
   Build local volatility surfaces using Black-76 futures option surfaces and product-specific calendars.

## 20. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_15_jump_diffusion_pide_pricer**  
   Extending PDE methods to jump-diffusion PIDEs.

2. **02_16_local_stochastic_volatility_model**  
   Combining local vol and stochastic vol.

3. **02_10_finite_difference_pde_pricer**  
   Modify the PDE pricer to use $\sigma_{\text{loc}}(t,S)$.

4. **03_09_volatility_surface_alpha_signals**  
   Use local-vol features as volatility surface signals.

5. **04_07_beta_weighted_tail_risk_hedging**  
   Stress test portfolios under surface-consistent volatility dynamics.

6. **07_02_volatility_surface_pricing_and_alpha**  
   Capstone combining surface construction, calibration, local vol, and alpha research.

## 21. Summary

This notebook implemented Dupire local volatility construction.

It showed how to:

1. generate a smooth implied volatility surface;
2. convert implied volatilities into call prices;
3. check call monotonicity and convexity;
4. compute numerical derivatives $C_T$, $C_K$, and $C_{KK}$;
5. apply Dupire's formula;
6. flag invalid local variance regions;
7. visualise the local volatility surface;
8. compare implied volatility and local volatility;
9. simulate paths under the local-volatility model;
10. validate approximate repricing against the input surface;
11. save diagnostics and manifests.

The key computational takeaway:

> Dupire is a differentiation problem. Smoothness, arbitrage constraints, and numerical stability matter as much as the formula.

The key financial takeaway:

> Local volatility can match vanilla surfaces, but it is not automatically a realistic model of future smile dynamics.

## 22. Further reading

- Dupire, B. "Pricing with a Smile."
- Derman and Kani on local volatility trees.
- Gatheral, *The Volatility Surface*.
- Andersen and Piterbarg, volatility modelling references.
- Literature on SVI/SSVI arbitrage-free surfaces.
- Literature on local-stochastic volatility calibration.
- Literature on local-volatility PDE and Monte Carlo methods.